# Tiny GPT End-to-End: Real Text, Tokenizers, and BPE

This notebook is a capstone for the Week 1-2 transformer internals work. You will train a small GPT-style language model on real public-domain text, starting from raw text and ending with generated samples.

What you will build:

1. A character tokenizer.
2. A byte-level BPE tokenizer from scratch.
3. A train/validation dataset of token sequences.
4. A tiny decoder-only Transformer.
5. A full training loop with evaluation, loss plotting, and text generation.

The goal is not to train a high-quality model. The goal is to connect every moving part: text -> tokens -> batches -> GPT -> loss -> optimizer -> generated text.

## How this notebook fits the roadmap

The earlier notebooks introduced attention, multi-head attention, positional encodings, normalization, FFNs, and a GPT decoder. This notebook uses those pieces in one real training pipeline.

You should be able to answer these questions by the end:

- Why tokenization is separate from model training.
- What BPE is optimizing.
- How a GPT training batch is created.
- Why the targets are the input tokens shifted by one position.
- How causal self-attention prevents future-token leakage.
- How loss curves and generated samples reveal whether learning is happening.

## 0. Setup

This notebook requires PyTorch. If the import fails, install it in the environment backing this notebook.

For CPU-only learning, the default model and training loop are intentionally small. If you have a GPU, you can increase `max_iters`, `n_layer`, `n_embd`, and `block_size` later.

In [ ]:
import math
import os
import random
import time
import urllib.request
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except ImportError as exc:
    raise ImportError(
        "PyTorch is required for this notebook. Install it with: pip install torch"
    ) from exc

seed = 1337
random.seed(seed)
torch.manual_seed(seed)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

## 1. Load real text

We will use Shakespeare text because it is public domain and small enough for a laptop experiment. The cell below tries to download the common Tiny Shakespeare dataset and caches it locally. If the network is unavailable, it falls back to an embedded public-domain excerpt repeated enough times to train a tiny model.

This is still real language: spelling, punctuation, repeated patterns, dialogue structure, and line breaks all matter.

In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
DATA_PATH = DATA_DIR / "tiny_shakespeare.txt"
DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

FALLBACK_TEXT = """
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us.
If they would yield us but the superfluity, while it were wholesome,
we might guess they relieved us humanely.
""".strip()

def load_text():
    if DATA_PATH.exists():
        print(f"Loading cached text from {DATA_PATH}")
        return DATA_PATH.read_text(encoding="utf-8")

    try:
        print("Downloading Tiny Shakespeare...")
        with urllib.request.urlopen(DATA_URL, timeout=10) as response:
            text = response.read().decode("utf-8")
        DATA_PATH.write_text(text, encoding="utf-8")
        return text
    except Exception as error:
        print(f"Download failed: {error}")
        print("Using embedded public-domain fallback text.")
        return (FALLBACK_TEXT + "\n\n") * 120

text = load_text()
print(f"Characters: {len(text):,}")
print(f"Unique characters: {len(set(text))}")
print("--- sample ---")
print(text[:800])

## 2. Tokenization: why it exists

Neural networks do not consume text directly. They consume integers. A tokenizer defines the mapping between strings and integer IDs.

A tokenizer has two core methods:

- `encode(text)`: convert a string into a list of token IDs.
- `decode(ids)`: convert token IDs back into a string.

Different tokenizers make different trade-offs. Character tokenization is easy and robust but produces long sequences. BPE learns frequent chunks, so common words or word pieces become single tokens.

## 3. Character tokenizer

The simplest tokenizer treats every unique character as a token. This is excellent for learning because the vocabulary is tiny and the logic is transparent.

However, character sequences are long. The phrase `Before we proceed` contains 17 characters including spaces, so a character model needs many time steps to represent simple words.

In [ ]:
class CharTokenizer:
    def __init__(self, training_text):
        self.chars = sorted(set(training_text))
        self.stoi = {char: idx for idx, char in enumerate(self.chars)}
        self.itos = {idx: char for char, idx in self.stoi.items()}
        self.vocab_size = len(self.chars)

    def encode(self, value):
        return [self.stoi[char] for char in value]

    def decode(self, ids):
        return "".join(self.itos[int(idx)] for idx in ids)

char_tokenizer = CharTokenizer(text)
sample = "Before we proceed any further"
sample_ids = char_tokenizer.encode(sample)

print(f"Character vocab size: {char_tokenizer.vocab_size}")
print(f"Sample: {sample!r}")
print(f"Encoded: {sample_ids}")
print(f"Decoded: {char_tokenizer.decode(sample_ids)!r}")

## 4. BPE intuition

Byte Pair Encoding learns a compact vocabulary by repeatedly merging the most frequent adjacent token pair.

Start with bytes:

```text
the theater
t h e   t h e a t e r
```

If `t h` appears often, merge it into a new token `th`. If `th e` appears often, merge it into `the`. Over many merges, frequent words and word fragments become shorter token sequences.

Modern LLM tokenizers often use byte-level BPE or related subword methods because byte-level tokenization can represent any text while still learning useful chunks.

## 5. Byte-level BPE from scratch

This implementation starts with the 256 possible byte values. Then it learns merges from the training text.

Important details:

- Base token IDs `0..255` are raw bytes.
- Each merge creates a new token ID.
- `encode` applies learned merges in training order.
- `decode` expands token IDs back into bytes and then decodes UTF-8.

This is intentionally educational rather than production optimized.

In [ ]:
def get_pair_counts(ids):
    return Counter(zip(ids, ids[1:]))

def merge_pair(ids, pair, new_id):
    merged = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            merged.append(new_id)
            i += 2
        else:
            merged.append(ids[i])
            i += 1
    return merged

class BytePairTokenizer:
    def __init__(self):
        self.merges = {}
        self.merge_ranks = {}
        self.vocab = {idx: bytes([idx]) for idx in range(256)}

    @property
    def vocab_size(self):
        return max(self.vocab.keys()) + 1

    def train(self, training_text, target_vocab_size=384, verbose=True):
        if target_vocab_size < 256:
            raise ValueError("target_vocab_size must be at least 256 for byte-level BPE")

        ids = list(training_text.encode("utf-8"))
        num_merges = target_vocab_size - 256

        for rank in range(num_merges):
            counts = get_pair_counts(ids)
            if not counts:
                break

            best_pair, best_count = counts.most_common(1)[0]
            if best_count < 2:
                break

            new_id = 256 + rank
            ids = merge_pair(ids, best_pair, new_id)
            self.merges[best_pair] = new_id
            self.merge_ranks[best_pair] = rank
            self.vocab[new_id] = self.vocab[best_pair[0]] + self.vocab[best_pair[1]]

            if verbose and (rank < 10 or (rank + 1) % 25 == 0):
                token_text = self.vocab[new_id].decode("utf-8", errors="replace")
                print(f"merge {rank + 1:3d}: {best_pair} -> {new_id:3d} | count={best_count:5d} | {token_text!r}")

        return ids

    def encode(self, value):
        ids = list(value.encode("utf-8"))

        while len(ids) >= 2:
            pairs = get_pair_counts(ids)
            eligible = [pair for pair in pairs if pair in self.merge_ranks]
            if not eligible:
                break

            best_pair = min(eligible, key=lambda pair: self.merge_ranks[pair])
            ids = merge_pair(ids, best_pair, self.merges[best_pair])

        return ids

    def decode(self, ids):
        raw = b"".join(self.vocab[int(idx)] for idx in ids)
        return raw.decode("utf-8", errors="replace")

    def show_merges(self, limit=20):
        rows = sorted(self.merges.items(), key=lambda item: self.merge_ranks[item[0]])[:limit]
        for pair, new_id in rows:
            token_text = self.vocab[new_id].decode("utf-8", errors="replace")
            print(f"{self.merge_ranks[pair]:3d}: {pair} -> {new_id:3d} | {token_text!r}")

## 6. Train the BPE tokenizer

The tokenizer is trained before the language model. Increasing `target_vocab_size` usually shortens sequences, but it also gives the model more output classes to predict.

For this notebook, `384` is a nice educational size: base bytes plus 128 learned merges.

In [ ]:
bpe_tokenizer = BytePairTokenizer()
bpe_ids_preview = bpe_tokenizer.train(text, target_vocab_size=384, verbose=True)

print(f"\nLearned merges: {len(bpe_tokenizer.merges)}")
print(f"BPE vocab size: {bpe_tokenizer.vocab_size}")
print(f"Original byte length: {len(text.encode('utf-8')):,}")
print(f"BPE token length after training merges: {len(bpe_ids_preview):,}")

In [ ]:
print("First learned merges:")
bpe_tokenizer.show_merges(limit=25)

sample = "Before we proceed any further, hear me speak."
char_ids = char_tokenizer.encode(sample)
bpe_ids = bpe_tokenizer.encode(sample)

print("\nSample text:")
print(sample)
print(f"Character tokens ({len(char_ids)}): {char_ids}")
print(f"BPE tokens       ({len(bpe_ids)}): {bpe_ids}")
print(f"Decoded BPE: {bpe_tokenizer.decode(bpe_ids)!r}")

## 7. Turn text into a language-modeling dataset

A decoder-only language model learns next-token prediction.

If the token sequence is:

```text
[10, 24, 83, 19, 42]
```

then one training example might be:

```text
x = [10, 24, 83, 19]
y = [24, 83, 19, 42]
```

The model sees `x` and is trained to predict `y` at every position. Causal masking ensures each position only uses previous tokens.

In [ ]:
all_ids = bpe_tokenizer.encode(text)
data = torch.tensor(all_ids, dtype=torch.long)

split_idx = int(0.9 * len(data))
train_data = data[:split_idx]
val_data = data[split_idx:]

print(f"Total BPE tokens: {len(data):,}")
print(f"Train tokens:     {len(train_data):,}")
print(f"Val tokens:       {len(val_data):,}")
print(f"Decoded preview: {bpe_tokenizer.decode(train_data[:80].tolist())!r}")

In [ ]:
block_size = 96
batch_size = 32

def get_batch(split):
    source = train_data if split == "train" else val_data
    if len(source) <= block_size + 1:
        raise ValueError("Dataset is too small for the chosen block_size. Reduce block_size.")

    starts = torch.randint(0, len(source) - block_size - 1, (batch_size,))
    x = torch.stack([source[start:start + block_size] for start in starts])
    y = torch.stack([source[start + 1:start + block_size + 1] for start in starts])
    return x.to(device), y.to(device)

xb, yb = get_batch("train")
print(f"x batch shape: {xb.shape}")
print(f"y batch shape: {yb.shape}")
print("\nFirst training example, x:")
print(bpe_tokenizer.decode(xb[0].tolist()))
print("\nFirst training example, y shifted by one token:")
print(bpe_tokenizer.decode(yb[0].tolist()))

## 8. Tiny GPT architecture

This is a small decoder-only Transformer. It uses:

- Token embeddings.
- Learned positional embeddings.
- Repeated Transformer blocks.
- Pre-norm LayerNorm.
- Causal multi-head self-attention.
- A GELU feed-forward network.
- A final language-modeling head.

This is intentionally close to the original GPT-style design. After this notebook, you can swap in RoPE, RMSNorm, and SwiGLU from the earlier notebooks.

In [ ]:
@dataclass
class TinyGPTConfig:
    vocab_size: int
    block_size: int = 96
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 128
    dropout: float = 0.1

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_head = config.n_head
        self.head_dim = config.n_embd // config.n_head

        self.qkv = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.proj = nn.Linear(config.n_embd, config.n_embd)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

        mask = torch.tril(torch.ones(config.block_size, config.block_size))
        self.register_buffer("causal_mask", mask.view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        batch, seq_len, channels = x.shape

        qkv = self.qkv(x)
        q, k, v = qkv.split(channels, dim=-1)

        q = q.view(batch, seq_len, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(batch, seq_len, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(batch, seq_len, self.n_head, self.head_dim).transpose(1, 2)

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(self.causal_mask[:, :, :seq_len, :seq_len] == 0, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        weights = self.attn_dropout(weights)

        out = weights @ v
        out = out.transpose(1, 2).contiguous().view(batch, seq_len, channels)
        out = self.resid_dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.GELU(),
            nn.Linear(4 * config.n_embd, config.n_embd),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln2 = nn.LayerNorm(config.n_embd)
        self.ffn = FeedForward(config)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

class TinyGPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.n_embd)
        self.position_embedding = nn.Embedding(config.block_size, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layer)])
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        batch, seq_len = idx.shape
        if seq_len > self.config.block_size:
            raise ValueError("Cannot forward sequence longer than block_size")

        positions = torch.arange(seq_len, device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(positions)[None, :, :]
        x = self.dropout(x)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=200, temperature=1.0, top_k=None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-8)

            if top_k is not None:
                values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < values[:, [-1]]] = float("-inf")

            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_idx], dim=1)
        return idx

## 9. Instantiate the model

This model is intentionally tiny. It has enough capacity to learn local spelling, punctuation, line breaks, and repeated phrase structure, but it will not produce fluent Shakespeare after a short CPU run.

In [ ]:
config = TinyGPTConfig(
    vocab_size=bpe_tokenizer.vocab_size,
    block_size=block_size,
    n_layer=4,
    n_head=4,
    n_embd=128,
    dropout=0.1,
)

model = TinyGPT(config).to(device)
num_params = sum(param.numel() for param in model.parameters())
print(config)
print(f"Parameters: {num_params:,}")

xb, yb = get_batch("train")
logits, loss = model(xb, yb)
print(f"Logits shape: {logits.shape}")
print(f"Initial loss: {loss.item():.4f}")
print(f"Expected random baseline: log(vocab_size) = {math.log(config.vocab_size):.4f}")

## 10. Generate before training

Before training, the model has random weights. The output should look mostly like noise. This is a useful sanity check: if the trained model later looks different, learning happened.

In [ ]:
prompt = "First Citizen:\n"
prompt_ids = torch.tensor([bpe_tokenizer.encode(prompt)], dtype=torch.long, device=device)
generated = model.generate(prompt_ids, max_new_tokens=200, temperature=1.0, top_k=50)
print(bpe_tokenizer.decode(generated[0].tolist()))

## 11. Training loop

The training loop repeatedly samples random chunks of tokens. For each batch:

1. Run the model forward.
2. Compute cross-entropy next-token loss.
3. Backpropagate gradients.
4. Update parameters with AdamW.

Validation loss is estimated every few iterations so you can see whether the model is improving on held-out text.

In [ ]:
@torch.no_grad()
def estimate_loss(eval_iters=20):
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for step in range(eval_iters):
            xb, yb = get_batch(split)
            _, loss = model(xb, yb)
            losses[step] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

learning_rate = 3e-4
max_iters = 600 if device == "cuda" else 250
eval_interval = 100
eval_iters = 25 if device == "cuda" else 10
grad_clip = 1.0

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.1)
history = []

start_time = time.time()
for iteration in range(max_iters + 1):
    if iteration % eval_interval == 0 or iteration == max_iters:
        losses = estimate_loss(eval_iters=eval_iters)
        elapsed = time.time() - start_time
        history.append({
            "iteration": iteration,
            "train": losses["train"],
            "val": losses["val"],
            "elapsed": elapsed,
        })
        print(
            f"iter {iteration:4d} | train loss {losses['train']:.4f} | "
            f"val loss {losses['val']:.4f} | elapsed {elapsed:.1f}s"
        )

    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()

## 12. Plot the loss curve

A healthy tiny run should show train loss decreasing. Validation loss may be noisy because the dataset and model are small. If validation loss rises while train loss falls, the model is overfitting.

In [ ]:
iterations = [row["iteration"] for row in history]
train_losses = [row["train"] for row in history]
val_losses = [row["val"] for row in history]

plt.figure(figsize=(8, 4))
plt.plot(iterations, train_losses, marker="o", label="train")
plt.plot(iterations, val_losses, marker="o", label="val")
plt.xlabel("iteration")
plt.ylabel("cross-entropy loss")
plt.title("Tiny GPT training curve")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 13. Generate after training

Generation is autoregressive. The model predicts one token, appends it to the context, then predicts the next token.

Try changing:

- `temperature`: lower is more conservative, higher is more random.
- `top_k`: limits sampling to the most likely tokens.
- `prompt`: conditions the model on a different prefix.

In [ ]:
model.eval()
prompts = [
    "First Citizen:\n",
    "All:\n",
    "Second Citizen:\n",
]

for prompt in prompts:
    prompt_ids = torch.tensor([bpe_tokenizer.encode(prompt)], dtype=torch.long, device=device)
    generated = model.generate(prompt_ids, max_new_tokens=350, temperature=0.85, top_k=50)
    print("=" * 80)
    print(f"PROMPT: {prompt!r}")
    print(bpe_tokenizer.decode(generated[0].tolist()))

## 14. Inspect what the model learned

Loss is useful, but generated text is the most intuitive diagnostic for language modeling.

Early in training, you usually see random characters or invalid-looking fragments. Later, the model often learns:

- Common punctuation.
- Spaces and line breaks.
- Speaker labels.
- Frequent words or word pieces.
- Local spelling patterns.

It will still fail at long-range coherence because this model is tiny and trained briefly.

In [ ]:
def sample_with_settings(prompt, temperatures=(0.6, 0.9, 1.2), top_k=50, max_new_tokens=180):
    prompt_ids = torch.tensor([bpe_tokenizer.encode(prompt)], dtype=torch.long, device=device)
    for temperature in temperatures:
        generated = model.generate(
            prompt_ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
        )
        print("=" * 80)
        print(f"temperature={temperature}, top_k={top_k}")
        print(bpe_tokenizer.decode(generated[0].tolist()))

sample_with_settings("First Citizen:\n")

## 15. Save a checkpoint

A useful checkpoint needs both the model weights and the tokenizer merges. Without the tokenizer, the integer IDs have no meaning.

This cell saves a compact checkpoint to `checkpoints/tiny_gpt_bpe.pt`.

In [ ]:
CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / "tiny_gpt_bpe.pt"

checkpoint = {
    "model_state_dict": model.state_dict(),
    "config": config.__dict__,
    "merges": {f"{a},{b}": new_id for (a, b), new_id in bpe_tokenizer.merges.items()},
    "merge_ranks": {f"{a},{b}": rank for (a, b), rank in bpe_tokenizer.merge_ranks.items()},
}

torch.save(checkpoint, CHECKPOINT_PATH)
print(f"Saved checkpoint to {CHECKPOINT_PATH}")

## 16. Experiments to try next

Use this notebook as a base and run controlled experiments. Change one variable at a time.

Tokenizer experiments:

1. Train with character tokens instead of BPE tokens. Compare sequence length and loss.
2. Try `target_vocab_size=512` or `target_vocab_size=768`. Does training improve?
3. Print the most common learned BPE tokens. Are they words, spaces plus words, or punctuation patterns?

Model experiments:

1. Replace learned positional embeddings with RoPE.
2. Replace LayerNorm with RMSNorm.
3. Replace the GELU FFN with SwiGLU.
4. Increase `n_layer` from 4 to 6.
5. Increase `block_size` from 96 to 128 or 256.

Training experiments:

1. Train longer and compare generated samples every 500 iterations.
2. Change learning rate to `1e-3`, `3e-4`, and `1e-4`.
3. Plot train vs validation loss and identify overfitting.

## Summary

You built the full tiny GPT pipeline:

1. Loaded real public-domain text.
2. Built a character tokenizer.
3. Built a byte-level BPE tokenizer from scratch.
4. Encoded text into token IDs.
5. Created next-token prediction batches.
6. Implemented a decoder-only Transformer.
7. Trained with cross-entropy loss and AdamW.
8. Generated samples before and after training.

This closes the Week 1-2 transformer internals loop: you now have the components and an end-to-end training run that shows how they work together.